# Demographic Dataset Processing

## Overview

This notebook processes a single demographic dataset obtained from the UK Census.

The objective is to clean, reorganise, and extract the required demographic information for the representative population generation framework.

The processed dataset will be exported as a standardised JSON file for later integration.

---

## Dataset Information

| Item | Description |
|------|-------------|
| Dataset |TS008_Sex |
| Source | Office for National Statistics (ONS) Census 2021 |
| Geography | local authorities: district / unitary (as of April 2023) |
| Input File |TS008_Sex|
| Output File | Sex_df|
| Notebook Author | Peter |
| Last Updated |2026-07-11 |

## 1. Import

In [1]:
# Import libraries

import pandas as pd
import numpy as np
import json

In [2]:
import sys
from pathlib import Path

# Find repository root (../..)
repo_root = Path.cwd().parents[1]

# Add repository root to Python import path
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [3]:
# Display settings

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Configuration

Specify the input dataset and output location.

In [4]:
# File paths
from pathlib import Path
input_file = "RM121_Sex.xlsx"
output_file = "sex.json"

raw_data_path = Path("../../data/demographic/raw/sex")
output_data_path = Path("../../data/demographic/processed/sex")

## 3. Load Dataset

In [5]:
df = pd.read_excel(raw_data_path / input_file)
df_processed = df.copy()

df_processed.head()

,Lower tier local authorities Code,Lower tier local authorities,Sex (2 categories) Code,Sex (2 categories),Age (86 categories) Code,Age (86 categories),Observation
0,E06000001,Hartlepool,1,Female,0,Aged under 1 year,449
1,E06000001,Hartlepool,1,Female,1,Aged 1 year,458
2,E06000001,Hartlepool,1,Female,2,Aged 2 years,510
3,E06000001,Hartlepool,1,Female,3,Aged 3 years,476
4,E06000001,Hartlepool,1,Female,4,Aged 4 years,518


## 4. Dataset Exploration

Inspect the structure of the dataset before processing.

In [6]:
df_processed.info()

df_processed.shape

df_processed.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56932 entries, 0 to 56931
Data columns (total 7 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   Lower tier local authorities Code  56932 non-null  object
 1   Lower tier local authorities       56932 non-null  object
 2   Sex (2 categories) Code            56932 non-null  int64 
 3   Sex (2 categories)                 56932 non-null  object
 4   Age (86 categories) Code           56932 non-null  int64 
 5   Age (86 categories)                56932 non-null  object
 6   Observation                        56932 non-null  int64 
dtypes: int64(3), object(4)
memory usage: 3.0+ MB


Index(['Lower tier local authorities Code', 'Lower tier local authorities',
       'Sex (2 categories) Code', 'Sex (2 categories)',
       'Age (86 categories) Code', 'Age (86 categories)', 'Observation'],
      dtype='object')

In [7]:
df_processed.describe(include="all")

,Lower tier local authorities Code,Lower tier local authorities,Sex (2 categories) Code,Sex (2 categories),Age (86 categories) Code,Age (86 categories),Observation
count,56932,56932,56932.000000,56932,56932.000000,56932,56932.000000
unique,331,331,NaN,2,NaN,86,NaN
top,W06000024,Merthyr Tydfil,NaN,Female,NaN,Aged under 1 year,NaN
freq,172,172,NaN,28466,NaN,662,NaN
mean,NaN,NaN,1.500000,NaN,42.500000,NaN,1046.819557
std,NaN,NaN,0.500004,NaN,24.824601,NaN,819.675209
min,NaN,NaN,1.000000,NaN,0.000000,NaN,0.000000
25%,NaN,NaN,1.000000,NaN,21.000000,NaN,559.000000
50%,NaN,NaN,1.500000,NaN,42.500000,NaN,792.000000
75%,NaN,NaN,2.000000,NaN,64.000000,NaN,1268.000000


## 5. Data Cleaning

Perform dataset-specific cleaning operations.

Typical tasks include:

- Rename variables
- Remove unnecessary columns
- Handle missing values
- Convert data types
- Standardise labels

In [8]:
# Cleaning code
selected_columns = ['Lower tier local authorities',
                    'Sex (2 categories)',
                    'Age (86 categories) Code',
                    'Observation']

df_processed = df_processed[selected_columns]

df_processed = df_processed.rename(columns={
    'Sex (2 categories)': 'Sex',
    'Age (86 categories) Code': 'Age',
    'Observation': 'Sex Population'
})

df_processed.head()

,Lower tier local authorities,Sex,Age,Sex Population
0,Hartlepool,Female,0,449
1,Hartlepool,Female,1,458
2,Hartlepool,Female,2,510
3,Hartlepool,Female,3,476
4,Hartlepool,Female,4,518


## 6. Feature Processing

Extract the demographic information required by the framework.

Examples include:

- Age groups
- Household composition
- Occupation
- Industry
- Car ownership
- Student status

The processing steps in this section depend on the current dataset.

In [9]:
# Dataset-specific processing
from ontology.AGE_ONTOLOGY import (
    AGE_GROUP,
    AGE_ORDER
)

In [10]:
def get_sex_distribution(local_authority):
    processed_data = df_processed[
        df_processed['Lower tier local authorities'] == local_authority].copy()

    processed_data['Age'] = processed_data['Age'].apply(AGE_GROUP)

    result = (processed_data.groupby(['Lower tier local authorities', 'Sex', 'Age'], as_index=False)['Sex Population'].sum())

    result['Age'] = pd.Categorical(result['Age'], categories=AGE_ORDER, ordered=True)

    result = result.sort_values('Age')

    result = result.reset_index(drop=True)

    return result

In [11]:
get_sex_distribution("Manchester")

,Lower tier local authorities,Sex,Age,Sex Population
0,Manchester,Female,0-4,17115
1,Manchester,Male,0-4,17263
2,Manchester,Female,5-11,25072
3,Manchester,Male,5-11,26277
4,Manchester,Female,12-17,20286
5,Manchester,Male,12-17,20939
6,Manchester,Female,18-22,32021
7,Manchester,Male,18-22,29762
8,Manchester,Female,23-34,62305
9,Manchester,Male,23-34,61136


## 7. Validation

Verify that the processed dataset is internally consistent before exporting.

In [12]:
# Examples

df_processed.head()

df_processed.info()

df_processed.isna().sum()

# Additional validation here

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56932 entries, 0 to 56931
Data columns (total 4 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   Lower tier local authorities  56932 non-null  object
 1   Sex                           56932 non-null  object
 2   Age                           56932 non-null  int64 
 3   Sex Population                56932 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 1.7+ MB


Lower tier local authorities    0
Sex                             0
Age                             0
Sex Population                  0
dtype: int64

## 8. Export Processed Dataset

In [13]:
output_data_path.mkdir(parents=True, exist_ok=True)

# Example

Sex_df = get_sex_distribution("Manchester")

Sex_df.to_json(
    output_data_path / output_file,
    orient="records",
    indent=4
)

## Summary

### Input

-

### Output

-

### Processing Completed

- Dataset loaded
- Dataset cleaned
- Features extracted
- Validation completed
- JSON exported